# Preparation

In [1]:
import pandas as pd
import sys
sys.path.append('../../')
from MEGA_utilities import data_col_standardize, data_remove_duplicate


# load data
Crick_H1N1 = pd.read_excel('../../../data/raw/data4model(Crick-H1N1).xlsx')
Crick_H3N2 = pd.read_excel('../../../data/raw/data4model(Crick-H3N2).xlsx')
origin_df = pd.concat([Crick_H1N1, Crick_H3N2]).reset_index(drop=True)

In [2]:
## select required columns
AA_data_filt1 = origin_df[['serumName','virusName','serumHA', 'serumNA', 'virusHA', 'virusNA', 
                           'serumPassCat','virusPassCat', 'ferret', 'serumType','HI_Dist']].copy()
## replace blank space in Names
AA_data_filt2 = AA_data_filt1.copy()
AA_data_filt2['serumName'] = AA_data_filt2['serumName'].str.replace(' ','')
AA_data_filt2['virusName'] = AA_data_filt2['virusName'].str.replace(' ','')
## remove *number in ferret column
AA_data_filt3 = AA_data_filt2.copy()
AA_data_filt3['ferret'] = AA_data_filt3['ferret'].str.replace(r'\*\d*', '', regex=True)
## remove duplicated row and mean HI_Dist
AA_data_filt4 = AA_data_filt3.groupby(['serumName', 'virusName', 'serumHA', 'serumNA', 'virusHA', 'virusNA', 
                                       'serumPassCat', 'virusPassCat', 'ferret']) \
        .agg({'serumType': 'first', 'HI_Dist': 'mean'}) \
        .reset_index()[['serumName', 'virusName', 'serumHA', 'serumNA', 'virusHA', 'virusNA',
                        'serumPassCat', 'virusPassCat', 'ferret', 'serumType', 'HI_Dist']]
## remove PassCat = 'BOTH'
AA_data_filt5 = AA_data_filt4[(AA_data_filt4['serumPassCat'] != 'BOTH') &
                              (AA_data_filt4['virusPassCat'] != 'BOTH')].reset_index(drop=True)
## replace PassCat to special token
AA_data_filt6 = AA_data_filt5.replace({'serumPassCat': {'EGG': '<EGG>', 'CELL': '<CELL>'},
                                       'virusPassCat': {'EGG': '<EGG>', 'CELL': '<CELL>'}})
## replace names by special token
unique_Virus_name = pd.concat([AA_data_filt6['serumName'], AA_data_filt6['virusName']]).unique()
name_map_dict = {key: value for key, value in zip(unique_Virus_name, 
                                                  ['<' + unique_Virus_name[i] + '>' for i in range(len(unique_Virus_name))])}
AA_data_filt7 = AA_data_filt6.copy()
AA_data_filt7['serumName'] = AA_data_filt7['serumName'].replace(name_map_dict)
AA_data_filt7['virusName'] = AA_data_filt7['virusName'].replace(name_map_dict)
## replace ferret by special token
unique_ferret_name = AA_data_filt7['ferret'].unique()
ferret_map_dict = {key: value for key, value in zip(unique_ferret_name,
                                                    ['<Ferret ' + str(i) + '>' for i in range(len(unique_ferret_name))])}
AA_data_filt7['ferret'] = AA_data_filt7['ferret'].replace(ferret_map_dict)

AA_data_final = AA_data_filt7.copy()

In [3]:
import json

with open('./name_dict_NPF.json', 'w') as file:
    json.dump(name_map_dict, file, indent=4)
with open('./ferret_dict_NPF.json', 'w') as file:
    json.dump(ferret_map_dict, file, indent=4)

In [4]:
vocal_path = './vocab_NPF.txt'

with open(vocal_path, 'a') as file:
    for value in name_map_dict.values():
        file.write(value + '\n')
    for value in ferret_map_dict.values():
        file.write(value + '\n')

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader

class AADataset(Dataset):
    def __init__(self, DataFrame):
        self.sequence = (DataFrame['serumHA'] + '<eos>' + DataFrame['serumNA'] + '<eos>' + DataFrame['virusHA'] + '<eos>' + DataFrame['virusNA'] + \
                         '<eos>' + DataFrame['serumPassCat'] + '<eos>' + DataFrame['virusPassCat'] + '<eos>' + DataFrame['ferret']).tolist()
        self.labels = torch.tensor(DataFrame['HI_Dist'].tolist())
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.sequence[idx], self.labels[idx]

In [6]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(AA_data_filt4, test_size=0.1, random_state=42)
train_df, valid_df = train_test_split(train_df, test_size=1/9, random_state=42)

train_dataset = AADataset(train_df)
valid_dataset = AADataset(valid_df)
test_dataset = AADataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [7]:
# train_df.to_csv('../../../data/processed/1.1/AA_train_df.csv', index=True)
# valid_df.to_csv('../../../data/processed/1.1/AA_valid_df.csv', index=True)
# test_df.to_csv('../../../data/processed/1.1/AA_test_df.csv', index=True)

In [8]:
from bio_tokenizer import BioTokenizer
from transformers import MegaConfig, MegaForSequenceClassification
from MEGA_utilities import count_parameters
from torch.optim import AdamW
from transformers import get_scheduler
import torch

# get tokenizer and model
tokenizer = BioTokenizer(vocab_file='./vocab_NPF.txt')

# update the num_vocab and num_label
config = MegaConfig()
config.num_labels=1
config.vocab_size=11841
config.max_positions=4000
config.num_attention_heads=4
config.num_hidden_layers=5
device = torch.device("cuda:1")
model = MegaForSequenceClassification(config)
model.to(device)
print("Number of parameters: %e"%count_parameters(model))

# optimizer
optimizer = AdamW(model.parameters(), lr=5e-4)

# scheduler
num_epochs = 160
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler(name="linear", optimizer=optimizer,
                             num_warmup_steps=len(train_loader), num_training_steps=num_training_steps)

Number of parameters: 2.647499e+06


In [9]:
from tqdm import tqdm
from utilities import print_exams
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
from utilities import EarlyStopping
from datetime import datetime

save_path = '../../../trained_model/1.4_Meta_info/NPF/'
progress_bar = tqdm(range(num_training_steps))
early_stopping = EarlyStopping(patience=10, delta=0.005, save_dir=save_path)

# Training loop
for epoch in range(num_epochs):
    model.train()
    loss_ls = []
    for batch_seq, batch_label in train_loader:
        batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt")
        batch_input = batch_input.to(device)
        batch_label = batch_label.to(device)

        outputs = model(**batch_input, labels=batch_label)

        loss = outputs.loss
        loss_ls.append(loss.item())

        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)
    train_loss = sum(loss_ls) / len(loss_ls)
    print('train loss :', train_loss)

    prediction_ls = []
    reference_ls = []
    logits_ls = []
    loss_ls_valid = []
    with torch.no_grad():
        model.eval()
        for batch_seq, batch_label in valid_loader:
            batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt")
            batch_input = batch_input.to(device)
            batch_label = batch_label.to(device)

            outputs = model(**batch_input, labels=batch_label)
            logits = outputs.logits
            loss = outputs.loss

            logits_ls.append(logits)
            loss_ls_valid.append(loss.item())
            prediction_ls += logits.tolist()
            prediction_ls_final = []
            for sublist in prediction_ls:
                for element in sublist:
                    prediction_ls_final.append(element)
            reference_ls += batch_label.tolist()

    print_exams(prediction_ls_final, reference_ls)
    valid_MAE = mean_absolute_error(reference_ls, prediction_ls_final)
    valid_mse = mean_squared_error(reference_ls, prediction_ls_final)
    valid_pearson = pearsonr(reference_ls, prediction_ls_final).statistic
    valid_spearman = spearmanr(reference_ls, prediction_ls_final).statistic
    
    early_stopping(valid_mse, model)
    if early_stopping.early_stop:
        print("Early stopping")
        break

    ## 将epoch信息写入log.txt
    with open(save_path + 'log.txt', 'a') as f:
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(f"[{current_time}] Epoch {epoch + 1}/{num_epochs}, train loss: {train_loss:.4f}, valid MAE: {valid_MAE:.4f}, valid MSE: {valid_mse:.4f}, valid Pearson: {valid_pearson:.4f}, valid Spearman: {valid_spearman:.4f}\n")

  1%|          | 8407/1345120 [25:12<62:48:55,  5.91it/s]

train loss : 2.918353812377561
MAE:  1.2371616034329271
MSE:  2.691822014042679
pearson correlation:  PearsonRResult(statistic=np.float64(0.5382731202872356), pvalue=np.float64(0.0))
spearman correlation:  SignificanceResult(statistic=np.float64(0.5731928998130738), pvalue=np.float64(0.0))
Validation MSE decrease (inf --> 2.691822).  Saving model ...


  1%|▏         | 16814/1345120 [52:26<60:11:20,  6.13it/s]  

train loss : 2.6067325796626672
MAE:  1.1134651913809253
MSE:  2.244887836695871
pearson correlation:  PearsonRResult(statistic=np.float64(0.6399247100998285), pvalue=np.float64(0.0))
spearman correlation:  SignificanceResult(statistic=np.float64(0.6414889957001721), pvalue=np.float64(0.0))
Validation MSE decrease (2.691822 --> 2.244888).  Saving model ...


  2%|▏         | 25221/1345120 [1:19:40<61:01:58,  6.01it/s] 

...

train loss : 0.7625557113488917


 43%|████▎     | 571677/1345120 [30:51:43<7783:06:43, 36.23s/it]

MAE:  0.6997728860530426
MSE:  0.8160164164543197
pearson correlation:  PearsonRResult(statistic=np.float64(0.8855976589078327), pvalue=np.float64(0.0))
spearman correlation:  SignificanceResult(statistic=np.float64(0.8536453489291963), pvalue=np.float64(0.0))
EarlyStopping counter: 9 out of 10


 43%|████▎     | 580083/1345120 [31:16:57<34:03:50,  6.24it/s]  

train loss : 0.760941638011165
MAE:  0.6980709281314319
MSE:  0.8235378994380878
pearson correlation:  PearsonRResult(statistic=np.float64(0.8854088808278775), pvalue=np.float64(0.0))
spearman correlation:  SignificanceResult(statistic=np.float64(0.8536350752700403), pvalue=np.float64(0.0))
EarlyStopping counter: 10 out of 10
Early stopping
